In [1]:
import logging

# pypdf에서 발생하는 WARNING 로그 숨기기
logging.getLogger("pypdf").setLevel(logging.ERROR)

In [2]:
# =============== 라이브러리 및 환경 설정 ===============
import os
import re
from pathlib import Path
from dotenv import load_dotenv
# PDF 로딩 및 청킹
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Retriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
# LLM 메시지
from langchain_core.messages import SystemMessage, HumanMessage


# =============== API 키 설정 ===============
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
# API 키 없이도 BM25 검색은 가능
print("OpenAI API 키 설정:", bool(OPENAI_API_KEY))


# =============== SPRi PDF 파일 로딩 ===============
# data 폴더에서 SPRi PDF 검색
pdf_candidates = sorted(
    (Path.cwd() / "data").glob("SPRi*.pdf")
)

# 정확히 하나의 PDF만 사용하도록 검사
if len(pdf_candidates) != 1:
    raise FileNotFoundError(
        f"SPRi PDF가 정확히 1개 필요합니다. "
        f"현재 발견된 파일: {pdf_candidates}"
    )

# 검색할 PDF 지정
pdf_path = pdf_candidates[0]

# PyPDFLoader: PDF를 페이지별 Document 객체로 변환
pdf_pages = PyPDFLoader(str(pdf_path)).load()

print("PDF 파일:", pdf_path.name)
print("PDF 페이지 수:", len(pdf_pages))


# =============== PDF 청킹 ===============
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

# 페이지별 문서를 작은 청크로 분할
pdf_chunks = pdf_splitter.split_documents(pdf_pages)

# 각 청크에 고유 ID 및 PDF 페이지 번호 저장
for i, doc in enumerate(pdf_chunks):
    doc.metadata["doc_id"] = f"SPRi-{i:04d}"

    # PyPDFLoader의 page는 0부터 시작
    # 사용자가 보는 PDF 페이지 번호는 1부터 시작
    doc.metadata["page_display"] = (
        int(doc.metadata.get("page", 0)) + 1
    )

print("생성된 청크 수:", len(pdf_chunks))


# =============== BM25 Retriever 생성 ===============
# BM25에서 사용할 간단한 토큰 분리 함수(한글, 영어, 숫자 단위로 분리)
def simple_tokens(text: str):
    return re.findall(r"[가-힣A-Za-z0-9]+", text.lower())

# PDF 청크만 대상으로 BM25 검색기 생성(API 키가 없어도 검색 가능)
pdf_bm25 = BM25Retriever.from_documents(
    pdf_chunks,
    preprocess_func=simple_tokens
)

# 질문마다 최대 4개 청크 반환
pdf_bm25.k = 4

print("BM25 Retriever 준비 완료")


# =============== Vector Retriever 및 LLM 생성 ===============
# API 키가 없을 경우 None으로 유지
pdf_vector_retriever = None
llm = None

if OPENAI_API_KEY:
    # OpenAI 임베딩 모델 생성: 텍스트를 벡터로 변환하여 의미 기반 검색 수행
    pdf_embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )

    # PDF 청크만 Chroma 벡터 DB에 저장
    pdf_vectorstore = Chroma.from_documents(
        documents=pdf_chunks,
        embedding=pdf_embedding_model,
        collection_name="spri_brief_practice"
    )

    # Vector Store를 Retriever로 변환 (상위 4개)
    pdf_vector_retriever = pdf_vectorstore.as_retriever(
        search_kwargs={"k": 4}
    )

    # PDF 검색 결과를 바탕으로 답변을 생성할 LLM
    llm = ChatOpenAI(
        model=MODEL_NAME,
        temperature=0
    )

    print("Vector Retriever 준비 완료")
    print("LLM 준비 완료")

else:
    # API 키가 없어도 BM25는 사용 가능
    print("API 키 없음: BM25 검색만 사용합니다.")


# =============== 자동 평가 질문 설정 ===============
evaluation_questions = [
    {
        "q": "EU AI 법 투명성 의무",
        "gold_page": 5
    },
    {
        "q": "Muse 개인용 AI 에이전트",
        "gold_page": 13
    },
    {
        "q": "2026년 9월 AI 모델 현황",
        "gold_page": 3
    }
]

print("평가 질문 수:", len(evaluation_questions))
print("\n실습문제 초기 설정 완료!")

C:\Users\sdesi\AppData\Local\Temp\ipykernel_11176\3146928904.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


OpenAI API 키 설정: True
PDF 파일: SPRi AI Brief 2026년 10월호(2026-10-09).pdf
PDF 페이지 수: 28
생성된 청크 수: 86
BM25 Retriever 준비 완료
Vector Retriever 준비 완료
LLM 준비 완료
평가 질문 수: 3

실습문제 초기 설정 완료!


___
### **4-5. (실습문제) SPRi PDF 기반 정책 산업 브리핑 어시스턴트**
AI 전략팀에서 2026년 10월 SPRi AI Brief를 바탕으로 회의 준비를 돕는 챗봇을 만듭니다. 팀원들은 PDF를 직접 읽지 않고도 관련 페이지를 찾아 요약하고 싶어 합니다.

1. 첨부된 PDF만 검색 대상으로 사용하세요. 외부 웹 검색은 사용하지 않습니다.
2. Retriever를 사용해 질문마다 관련 청크 최대 4개를 가져오세요.
3. 답변에는 PDF 페이지 번호를 포함하세요.
4. API 키가 없을 때에도 검색 결과 페이지는 확인할 수 있어야 합니다.
5. 답변 근거가 없으면 추측하지 말고 확인 불가라고 답하세요.
6. 두 개 이상의 질문을 자동 테스트하고, 결과를 비교하세요.
7. BM25와 벡터 Retriever를 교체해 결과가 어떻게 다른지 확인하세요.

In [3]:
# 사람인지, AI인지 구분하는 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate

In [4]:
# SPRi PDF에서 답변을 생성하는 함수
def answer_from_spri_pdf(question, retriever):

    docs = retriever.invoke(question)[:4]

    if not docs:
        return {"answer": "PDF에서 관련 내용을 확인할 수 없습니다.", "sources": []}
    
    pages = sorted(set(doc.metadata.get("page_display", doc.metadata.get("page", 0) + 1) for doc in docs))
    context = "\n\n".join(
        f"[PDF {doc.metadata.get('page_display', doc.metadata.get('page', 0) + 1)}쪽]\n"
        f"{doc.page_content}"
        for doc in docs
    )
    if not OPENAI_API_KEY:
        return {"answer": "API KEY ERROR!"}

    prompt = ChatPromptTemplate.from_messages([("system", """
        당신은 SPRi AI 정책 산업 브리핑 어시스턴트입니다.

        반드시 제공된 PDF 내용만 근거로 답변하세요.
        외부 지식이나 추측을 사용하지 마세요.
        근거가 부족하면 '제공된 PDF에서 확인할 수 없습니다.'라고 답하세요.
        답변에는 근거가 되는 PDF 페이지 번호를 포함하세요.

        PDF 내용:
        {context}
    """), ("human", "{question}")])

    llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})

    return {"answer": response.content, "sources": pages}

In [5]:
# SPRi PDF 정책 산업 브리핑 챗봇
def spri_chatbot():
    print("SPRi PDF 정책 산업 브리핑 챗봇")
    print("종료하려면 '종료'를 입력하세요.")

    # ★ 추가
    question_count = 0

    while True:
        question = input("\n질문을 입력하세요: ").strip()
        # ★ 추가
        question_count += 1

        # ★ 추가: 몇번째 질문인지, 질문내용 출력
        print(f"\n질문 {question_count}: {question}")

        if question.lower() in ['종료', 'exit', 'quit']:
            print("챗봇을 종료합니다.")
            break

        if not question:
            continue

        print("\n검색 방법을 선택하세요.")
        print("1. BM25")
        print("2. Vector")

        method = input("선택(1/2): ").strip()

        if method == "2":
            if not OPENAI_API_KEY:
                print("API 키가 없어 BM25로 검색합니다.")
                retriever = pdf_bm25
            else:
                retriever = pdf_vector_retriever
        else:
            retriever = pdf_bm25

        result = answer_from_spri_pdf(question, retriever=retriever)

        print("\n[챗봇 답변]")
        print(result['answer'])

        print("\n[참고 PDF 페이지]")
        print(result['sources'])

In [ ]:
spri_chatbot()

SPRi PDF 정책 산업 브리핑 챗봇
종료하려면 '종료'를 입력하세요.

질문 1: 

질문 2: 

질문 3: 

질문 4: ㄳㄷ

검색 방법을 선택하세요.
1. BM25
2. Vector

[챗봇 답변]
무엇을 도와드릴까요? 제공된 PDF 내용에 관한 질문을 입력해 주세요.

[참고 PDF 페이지]
[26, 27]
